<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 5 · Money Markets and Short-Rate Instruments

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch05_money_markets_short_rate_instruments.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

The short end of the curve is where monetary policy, funding, and derivative discounting meet, and where small convention choices matter most. Chapter 5 converts money-market quotes into discount factors under simple compounding, compounds overnight fixings one accrual day at a time on an ACT/360 basis, and derives forward term rates from two discount factors, with deposits, overnight indexed swaps, forward-rate agreements, futures, and repo placed on one instrument ladder.

Afterwards you can turn a quoted deposit rate plus a day-count basis into a discount factor, and explain why an overnight index is built day by day rather than quoted as a single period rate.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script converts the two illustrative deposit quotes (3M at 3.8%, 6M at 4.0%, simple compounding) into discount factors, compounds the four daily overnight fixings on an ACT/360 basis into a single period return, and derives the 3M-into-6M forward rate from the two discount factors. In the printed summary, check that the forward 0.041605 exceeds both quoted deposits, and that the tiny overnight return is a four-day figure, not an annual rate.

In [ ]:
SCRIPT = ROOT / "code" / "ch05_money_markets.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Check each printed number against the convention that produced it.

- Confirm the discount factors follow simple compounding: the 3M quote gives 1 / (1 + 0.038 * 0.25) = 0.990589 and the 6M quote gives 1 / (1 + 0.040 * 0.50) = 0.980392.
- Read `forward_3m_6m` 0.041605 as (D_3M / D_6M - 1) / 0.25: with an upward-sloping short end, the forward must exceed both deposit quotes.
- Treat `overnight_return` 0.00042423 as a four-day compounded return, not an annual rate: annualized simply over the elapsed 4/360 of a year it is about 3.82%.
- Keep the provenance split: the quotes and overnight fixings are illustrative teaching files, while the Euro short-term rate (ESTR) and New York Fed reference-rate CSVs in `data/` are official snapshots with metadata sidecars.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch05_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: Compound an overnight rate over four days

An overnight index is built one accrual day at a time, never as a single quoted period rate. The mini-lab compounds four illustrative daily fixings (3.80%, 3.82%, 3.81%, 3.84%) on an ACT/360 basis, where each day contributes a gross factor of one plus the rate times 1/360, and converts the compounded return into a simple annualized equivalent. All fixings are deterministic teaching inputs.

In [ ]:
import numpy as np

fixings = np.array([0.0380, 0.0382, 0.0381, 0.0384])  # daily rates
tau = np.full(fixings.size, 1.0 / 360.0)  # ACT/360 day fractions
factor = np.prod(1.0 + fixings * tau)  # compound growth factor
period_return = factor - 1.0  # compounded four-day return
annualized = period_return / tau.sum()  # simple annual equivalent
print(f"compound factor: {factor:.8f}")
print(f"period return: {period_return:.8f}")
print(f"annualized rate: {annualized:.6f}")

### What to observe

With these inputs the calculation yields a compound factor of 1.00042423, a four-day return of 0.00042423 (4.2423 basis points), and a simple annualized rate of 0.038181, matching the script's `overnight_return` to eight decimals. Compounding puts the annualized figure slightly above the 3.8175% simple average of the fixings, because each day's interest earns interest on the following days.

## Exercises

Use the mini-lab fixings and the chapter script outputs to complete these checks, naming the day-count basis with every rate.

1. **Add a fifth fixing.** Extend the array with a 3.86% fixing and recompute the compound factor and annualized rate. *Hint:* Add one element to `fixings`; `tau` sizes itself from the array.
2. **Change the day-count basis.** Recompute the period return on an ACT/365 basis and explain what happens to the annualized equivalent. *Hint:* The period return falls to about 0.00041842, while the annualized rate stays near 0.038181 because the year fraction falls proportionally.
3. **Convert the script's quotes.** Turn the 3M and 6M quotes into discount factors and verify 0.990589 and 0.980392. *Hint:* Simple compounding gives 1 / (1 + rate * tau).
4. **Extract the forward.** Compute the 3M-into-6M forward from those two discount factors and confirm the script's 0.041605. *Hint:* `(d_3m / d_6m - 1) / 0.25` over the 0.25-year forward interval.
5. **Annualize correctly.** Explain why the four-day return 0.00042423 annualizes to about 3.82% rather than to about 15.3%. *Hint:* Divide by the elapsed year fraction `tau.sum()` (4/360); multiplying the return by 360 ignores that it already spans four days.

For more detail, see Chapter 5 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">